<a href="https://colab.research.google.com/github/johnphilippowell/get-started-with-data-engineering-on-databricks-repo-example/blob/published/earth_engine_notebooks/dynamic_world_europe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap.core as geemap
import time

# Trigger the authentication flow.
ee.Authenticate()

project_id = 'fire-blocks-386109'

# set ouput project
ee.Initialize(project=project_id)

In [ ]:
# get a list of countries
print(ee.FeatureCollection('FAO/GAUL/2015/level0').aggregate_array('ADM0_NAME').getInfo())

['Montenegro', 'Serbia', 'South Sudan', 'Sudan', 'Taiwan', 'Cocos (Keeling) Islands', 'Christmas Island', 'Ashmore and Cartier Islands', 'Faroe Islands', 'Mayotte', 'Réunion', 'Tromelin Island', 'Juan de Nova Island', 'Glorioso Island', 'Europa Island', 'Bassas da India', 'Saint Pierre et Miquelon', 'French Southern and Antarctic Territories', 'Denmark', 'The former Yugoslav Republic of Macedonia', 'Croatia', 'Malta', 'San Marino', 'Slovenia', 'Greece', 'Italy', 'Portugal', 'Spain', 'Bosnia and Herzegovina', 'Andorra', 'Albania', 'Monaco', 'Netherlands', 'Luxembourg', 'Liechtenstein', 'France', 'Germany', 'Switzerland', 'Belgium', 'Austria', 'Australia', 'Palau', 'Canada', 'Canada', 'Canada', 'Mozambique', 'Mauritius', 'Malawi', 'Rwanda', 'Somalia', 'Zambia', 'Kenya', 'Madagascar', 'Seychelles', 'United Republic of Tanzania', 'Uganda', 'Zimbabwe', 'Ethiopia', 'Eritrea', 'Djibouti', 'Comoros', 'Burundi', 'Gabon', 'Equatorial Guinea', 'Chad', 'Sao Tome and Principe', 'Congo', 'Democratic

In [ ]:
def run_urban_masks (state):
  try:
    scale = 25
    year = 2025
    gt = 25
    output_name = "urban_area_{}_{}_{}".format(scale, year, gt)
    bigquery_name = "fire-blocks-386109.urban_masks_europe.{}".format(output_name)

    print(bigquery_name)
    states = ee.FeatureCollection("FAO/GAUL/2015/level0")

    # get a list of countries
    # print(ee.FeatureCollection('FAO/GAUL/2015/level0').aggregate_array('ADM0_NAME'))

    # buffer to pick up sea areas which are missed by fao boundary. galicia was terrible
    filtered = states.filter(ee.Filter.eq('ADM0_NAME', state))
    geometry = filtered.geometry().simplify(200).buffer(500)

    startDate = '2024-06-01'
    endDate = '2025-06-01'

    dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
        .filterDate(startDate, endDate) \
        .filterBounds(geometry)

    # select buildings band, reduce by mean, get those above gt threshold (25%), update mask
    buildings_prob = dw.select('built')
    meanProbability = buildings_prob.reduce(ee.Reducer.mean())
    builtGt25 = meanProbability.gte(gt/100.).toFloat()
    filteredBuiltGt25 = meanProbability.updateMask(builtGt25).toInt()

    # create a 500km grid, to subdivide jobs to prevent out of memory errors
    grid = geometry.coveringGrid('epsg:4326', 500000)
    aoi = ee.Feature(geometry)
    gridClipped = grid.map(lambda feat: aoi.intersection(feat, 0.1))

    # create a list of grid objects, so we can loop through them (yuk)
    list = gridClipped.toList(gridClipped.size())
    sz = list.size().getInfo()

    # loop through the grid collection, reduce, export
    for x in range(0, sz):
      print(ee.Feature(list.get(x)).geometry().area().getInfo())
      geom = ee.Feature(list.get(x)).geometry()
      vectors = filteredBuiltGt25.reduceToVectors(**{
          'geometry': geom,
          'scale': scale,
          'geometryType': 'polygon',
          'eightConnected': False,
          'maxPixels': 1e12,
          'bestEffort': True
      })

      # simplify vectors to same scale as reduction, add country column and remove unnecessary columns
      vectors = vectors.map(lambda feat: feat.simplify(scale))
      vectorsWithCountry = vectors.map(lambda feat: feat.set('country', state))
      propertiesToKeep = ['country', 'geo']
      vectorsWithCountryOut = vectorsWithCountry.select(propertiesToKeep)

      # export to BigQuery
      task = ee.batch.Export.table.toBigQuery(**{
        'collection': vectorsWithCountryOut,
        'description':  "{}_{}".format(state, x),
        'table': bigquery_name,
        'append': True,
        'maxVertices': 1e8
      })

      task.start()

      while task.active():
        print('Polling for task (id: {}).'.format(task.id))
        time.sleep(30)

    """
    m = geemap.Map(**{'height': '800px'})
    m.add_layer(filteredBuiltGt25.clip(geometry), {'min': 0, 'max': 1}, 'Urban max')

    m.center_object(geometry, 7);
    m.add_layer(grid, {}, "country outline")
    m.add_layer(gridClipped, {}, "grid clipped")

    m
    """
  except Exception as e:
    # generally caused by no grid/outline data -- Kosovo, for example
    print(e)
    pass




In [ ]:
for state in ["North Macedonia", "Romania", "San Marino", "Serbia", "Slovakia", "Slovenia", "Switzerland"]:
  print(f"running {state}")
  run_urban_masks(state)

running North Macedonia
fire-blocks-386109.urban_masks_europe.urban_area_25_2025_25
Collection.toList: The value of 'count' must be positive. Got: 0.
running Romania
fire-blocks-386109.urban_masks_europe.urban_area_25_2025_25
2535431091.295477
Polling for task (id: H7FLY4OH65NSF4OC7W3P5EAX).
Polling for task (id: H7FLY4OH65NSF4OC7W3P5EAX).
Polling for task (id: H7FLY4OH65NSF4OC7W3P5EAX).
41675641708.944115
Polling for task (id: ND5GPPXUI6NVQPLNZXXRE4WE).
Polling for task (id: ND5GPPXUI6NVQPLNZXXRE4WE).
Polling for task (id: ND5GPPXUI6NVQPLNZXXRE4WE).
Polling for task (id: ND5GPPXUI6NVQPLNZXXRE4WE).
Polling for task (id: ND5GPPXUI6NVQPLNZXXRE4WE).
16639279399.807487
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
Polling for task (id: W5NPG6F2RPUOUG6AY52D6JUQ).
28209722649.79357
Polling for ta